## Objective

The objetive of this notebook is to create features for the forecasting model.

In this phase, the dataset will be prepared for supervised learning by creating variables that help explain future demand. The target variable remains `quantity_sold`.

Special care will be taken to avoid **data leakage**, ensuring that all features are created only with information availiable before the prediction date.

The output of this notebook will be a feature-ready dataset for the modeling phase.

In [1]:
import os 
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path("..").resolve()
os.chdir(ROOT)

DATA_PATH = Path("data/processed/daily_product_sales.csv")

df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
print("columns:", list(df.columns))
print(f"Date range: {df['sale_date'].min()} to {df['sale_date'].max()}")
df.head()

shape: (6168, 9)
columns: ['product_name', 'sale_date', 'quantity_sold', 'total_revenue_brl', 'estimated_profit_brl', 'discount_pct', 'unit_price_brl', 'current_stock_snapshot', 'weather_condition']
Date range: 2025-01-01 to 2026-05-29


,product_name,sale_date,quantity_sold,total_revenue_brl,estimated_profit_brl,discount_pct,unit_price_brl,current_stock_snapshot,weather_condition
0,Bag Delivery 45L,2025-01-01,5.0,889.97,414.97,7.5,178.1725,84.0,Chuva Forte
1,Intercomunicador,2025-01-01,0.0,0.00,0.00,NaN,NaN,NaN,NaN
2,Carregador USB Moto,2025-01-01,0.0,0.00,0.00,NaN,NaN,NaN,NaN
3,Luva Motoboy,2025-01-01,10.0,777.04,527.04,5.0,78.2460,71.2,Ensolarado
4,Capacete Pro Tork,2025-01-01,7.0,2265.01,1005.01,5.0,326.7925,65.5,Nublado


## Creating Date_Based Features

In [2]:
df_features = df.copy()
df_features = df_features.sort_values(["product_name", "sale_date"])
df_features["sale_date"] = pd.to_datetime(df_features["sale_date"])

df_features["day_of_week"] = df_features["sale_date"].dt.dayofweek
df_features["day_of_month"] = df_features["sale_date"].dt.day
df_features["month"] = df_features["sale_date"].dt.month
df_features["week_of_year"] = df_features["sale_date"].dt.isocalendar().week.astype(int)
df_features["is_weekend"] = df_features["day_of_week"].isin([5,6]).astype(int)

df_features.head(5)

,product_name,sale_date,quantity_sold,total_revenue_brl,estimated_profit_brl,discount_pct,unit_price_brl,current_stock_snapshot,weather_condition,day_of_week,day_of_month,month,week_of_year,is_weekend
0,Bag Delivery 45L,2025-01-01,5.0,889.97,414.97,7.5,178.1725,84.000000,Chuva Forte,2,1,1,1,0
16,Bag Delivery 45L,2025-01-02,2.0,364.80,174.80,0.0,182.4000,53.000000,Chuva Leve,3,2,1,1,0
31,Bag Delivery 45L,2025-01-03,1.0,183.92,88.92,5.0,183.9200,24.000000,Ensolarado,4,3,1,1,0
43,Bag Delivery 45L,2025-01-04,3.0,536.25,251.25,5.0,178.7500,76.333333,Nublado,5,4,1,1,1
52,Bag Delivery 45L,2025-01-05,8.0,1442.82,682.82,4.0,179.0000,97.600000,Chuva Leve,6,5,1,1,1


## Creating Lag Features

In [3]:
df_features["quantity_1d"] =  (
    df_features
    .groupby("product_name")["quantity_sold"]
    .shift(1)
)

df_features["quantity_7d"] =  (
    df_features
    .groupby("product_name")["quantity_sold"]
    .shift(7)
)

df_features["quantity_14d"] =  (
    df_features
    .groupby("product_name")["quantity_sold"]
    .shift(14)
)
df_features

,product_name,sale_date,quantity_sold,total_revenue_brl,estimated_profit_brl,discount_pct,unit_price_brl,current_stock_snapshot,weather_condition,day_of_week,day_of_month,month,week_of_year,is_weekend,quantity_1d,quantity_7d,quantity_14d
0,Bag Delivery 45L,2025-01-01,5.0,889.97,414.97,7.500000,178.172500,84.000000,Chuva Forte,2,1,1,1,0,NaN,NaN,NaN
16,Bag Delivery 45L,2025-01-02,2.0,364.80,174.80,0.000000,182.400000,53.000000,Chuva Leve,3,2,1,1,0,5.0,NaN,NaN
31,Bag Delivery 45L,2025-01-03,1.0,183.92,88.92,5.000000,183.920000,24.000000,Ensolarado,4,3,1,1,0,2.0,NaN,NaN
43,Bag Delivery 45L,2025-01-04,3.0,536.25,251.25,5.000000,178.750000,76.333333,Nublado,5,4,1,1,1,1.0,NaN,NaN
52,Bag Delivery 45L,2025-01-05,8.0,1442.82,682.82,4.000000,179.000000,97.600000,Chuva Leve,6,5,1,1,1,3.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6113,Suporte Celular Moto,2026-05-25,4.0,233.12,161.12,0.000000,58.280000,55.250000,Nublado,0,25,5,22,0,1.0,1.0,2.0
6121,Suporte Celular Moto,2026-05-26,3.0,179.94,125.94,0.000000,60.115000,83.000000,Ensolarado,1,26,5,22,0,4.0,5.0,3.0
6134,Suporte Celular Moto,2026-05-27,4.0,233.61,161.61,1.666667,58.573333,93.000000,Ensolarado,2,27,5,22,0,3.0,4.0,5.0
6152,Suporte Celular Moto,2026-05-28,2.0,117.68,81.68,2.500000,58.840000,99.000000,Ensolarado,3,28,5,22,0,4.0,1.0,1.0


## Create Rolling Window Features

In [4]:
df_features["rolling_1d"] = (
    df_features
    .groupby("product_name")["quantity_sold"]
    .transform(lambda x: x.shift(1).rolling(window=1).mean())
)

df_features["rolling_7d"] = (
    df_features
    .groupby("product_name")["quantity_sold"]
    .transform(lambda x: x.shift(1).rolling(window=7).mean())
)

df_features["rolling_14d"] = (
    df_features
    .groupby("product_name")["quantity_sold"]
    .transform(lambda x: x.shift(1).rolling(window=14).mean())
)

df_features

,product_name,sale_date,quantity_sold,total_revenue_brl,estimated_profit_brl,discount_pct,unit_price_brl,current_stock_snapshot,weather_condition,day_of_week,day_of_month,month,week_of_year,is_weekend,quantity_1d,quantity_7d,quantity_14d,rolling_1d,rolling_7d,rolling_14d
0,Bag Delivery 45L,2025-01-01,5.0,889.97,414.97,7.500000,178.172500,84.000000,Chuva Forte,2,1,1,1,0,NaN,NaN,NaN,NaN,NaN,NaN
16,Bag Delivery 45L,2025-01-02,2.0,364.80,174.80,0.000000,182.400000,53.000000,Chuva Leve,3,2,1,1,0,5.0,NaN,NaN,5.0,NaN,NaN
31,Bag Delivery 45L,2025-01-03,1.0,183.92,88.92,5.000000,183.920000,24.000000,Ensolarado,4,3,1,1,0,2.0,NaN,NaN,2.0,NaN,NaN
43,Bag Delivery 45L,2025-01-04,3.0,536.25,251.25,5.000000,178.750000,76.333333,Nublado,5,4,1,1,1,1.0,NaN,NaN,1.0,NaN,NaN
52,Bag Delivery 45L,2025-01-05,8.0,1442.82,682.82,4.000000,179.000000,97.600000,Chuva Leve,6,5,1,1,1,3.0,NaN,NaN,3.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6113,Suporte Celular Moto,2026-05-25,4.0,233.12,161.12,0.000000,58.280000,55.250000,Nublado,0,25,5,22,0,1.0,1.0,2.0,1.0,2.142857,2.428571
6121,Suporte Celular Moto,2026-05-26,3.0,179.94,125.94,0.000000,60.115000,83.000000,Ensolarado,1,26,5,22,0,4.0,5.0,3.0,4.0,2.571429,2.571429
6134,Suporte Celular Moto,2026-05-27,4.0,233.61,161.61,1.666667,58.573333,93.000000,Ensolarado,2,27,5,22,0,3.0,4.0,5.0,3.0,2.285714,2.571429
6152,Suporte Celular Moto,2026-05-28,2.0,117.68,81.68,2.500000,58.840000,99.000000,Ensolarado,3,28,5,22,0,4.0,1.0,1.0,4.0,2.285714,2.500000


## Discovering and Handling the NaN values

In [5]:
nan_summary = df_features.isna().sum()

nan_df = (
    nan_summary[nan_summary > 0]
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={
        "index": "Features",
        0: "Missing Values (NaN)"
    })
)

nan_df

,Features,Missing Values (NaN)
0,discount_pct,2707
1,unit_price_brl,2707
2,current_stock_snapshot,2707
3,weather_condition,2707
4,quantity_14d,168
5,rolling_14d,168
6,quantity_7d,84
7,rolling_7d,84
8,quantity_1d,12
9,rolling_1d,12


## Handling Time/History Features
- For this features i will drop all `NaN` data
- They are expected beacause the first days **do not have enough historical data**

In [6]:
history_features = [
    "quantity_1d",
    "quantity_7d",
    "quantity_14d",
    "rolling_7d",
    "rolling_14d"
]

df_features = df_features.dropna(subset=history_features)

## Handling Unit/Business Numeric Features
- For this features i will have a **different handling:**
  - `discount_pct`:  I will aplly **no discount (0)**,
  - `current_stock_snapshot` and `unit_price_brl`; I will fill **using the last known value**.
  

In [7]:
df_features["discount_pct"] = df_features["discount_pct"].fillna(0)
df_features["current_stock_snapshot"] = (
    df_features
    .sort_values(["product_name", "sale_date"])
    .groupby("product_name")["current_stock_snapshot"]
    .ffill()
)
df_features["unit_price_brl"] = (
    df_features
    .sort_values(["product_name", "sale_date"])
    .groupby("product_name")["unit_price_brl"]
    .ffill()
)
df_features["current_stock_snapshot"] = df_features["current_stock_snapshot"].fillna(
    df_features.groupby("product_name")["current_stock_snapshot"].transform("median")
)
df_features["unit_price_brl"] = df_features["unit_price_brl"].fillna(
    df_features.groupby("product_name")["unit_price_brl"].transform("median")
)

C:\Users\henrique.nascimento\AppData\Local\Temp\ipykernel_27028\414224633.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_features["discount_pct"] = df_features["discount_pct"].fillna(0)
C:\Users\henrique.nascimento\AppData\Local\Temp\ipykernel_27028\414224633.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_features["current_stock_snapshot"] = (
C:\Users\henrique.nascimento\AppData\Local\Temp\ipykernel_27028\414224633.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

## Categorical Feature
- For `weather_condition` i will treat as a **date-level feature**. So i will return the same weather in the same date.

In [8]:
weather_by_date = (
    df
    .copy()
    .assign(sale_date=lambda x: pd.to_datetime(x["sale_date"]))
    .sort_values("sale_date")
    .groupby("sale_date", as_index=False)
    .agg(weather_condition=("weather_condition", "last"))
)

df_features = df_features.drop(columns=["weather_condition"], errors="ignore")

df_features = df_features.merge(
    weather_by_date,
    on="sale_date",
    how="left"
)


In [9]:
handled_nan_df = (
    df_features
    .isna()
    .sum()
    .reset_index()
    .rename(columns={
        "index": "Features",
        0: "Missing Values (NaN)"
    })
)

handled_nan_df

,Features,Missing Values (NaN)
0,product_name,0
1,sale_date,0
2,quantity_sold,0
3,total_revenue_brl,0
4,estimated_profit_brl,0
5,discount_pct,0
6,unit_price_brl,0
7,current_stock_snapshot,0
8,day_of_week,0
9,day_of_month,0


## Removing Leakage Columns

In [10]:
leakage_cols = [
    "total_revenue_brl",
    "estimated_profit_brl",
    "discount_pct",
    "current_stock_snapshot"
]

df_features = df_features.drop(columns=leakage_cols)

## Validating Final Feature Dataset

In [11]:
print("shape:", df_features.shape)
print("columns:", list(df_features.columns))
df_features.head()

shape: (6000, 16)
columns: ['product_name', 'sale_date', 'quantity_sold', 'unit_price_brl', 'day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend', 'quantity_1d', 'quantity_7d', 'quantity_14d', 'rolling_1d', 'rolling_7d', 'rolling_14d', 'weather_condition']


,product_name,sale_date,quantity_sold,unit_price_brl,day_of_week,day_of_month,month,week_of_year,is_weekend,quantity_1d,quantity_7d,quantity_14d,rolling_1d,rolling_7d,rolling_14d,weather_condition
0,Bag Delivery 45L,2025-01-15,5.0,179.1025,2,15,1,3,0,7.0,6.0,5.0,7.0,4.428571,4.071429,Ensolarado
1,Bag Delivery 45L,2025-01-16,4.0,179.2300,3,16,1,3,0,5.0,3.0,2.0,5.0,4.285714,4.071429,Ensolarado
2,Bag Delivery 45L,2025-01-17,4.0,179.7700,4,17,1,3,0,4.0,2.0,1.0,4.0,4.428571,4.214286,Chuva Leve
3,Bag Delivery 45L,2025-01-18,3.0,175.6100,5,18,1,3,1,4.0,4.0,3.0,4.0,4.714286,4.428571,Ensolarado
4,Bag Delivery 45L,2025-01-19,0.0,175.6100,6,19,1,3,1,3.0,5.0,8.0,3.0,4.571429,4.428571,Nublado


## Saving Modeling Dataset

In [12]:
feature_path = ROOT / "data" / "processed" / "modeling_dataset.csv"
feature_path.parent.mkdir(parents=True, exist_ok=True)
df_features.to_csv(feature_path, index=False, encoding="utf-8")
print("Saved the processed dataset to:", feature_path.resolve())

Saved the processed dataset to: C:\DEV\motostock-ai\data\processed\modeling_dataset.csv


## Feature Engineering Summary

In this notebook, the prepared time series dataset was transformed into a feature-ready dataset for supervised learning.

The main features created were:

* Date-based features, such as `day_of_week`, `day_of_month`, `month`, `week_of_year` and `is_weekend`.
* Historical demand features, such as `quantity_1d`, `quantity_7d` and `quantity_14d`.
* Rolling window features, such as `rolling_7d` and `rolling_14d`, created using only past demand values.

Missing values were handled according to the meaning of each feature. Historical features with missing values were removed because the first observations of each product do not have enough past information. Numeric business features were handled using simple rules, such as filling missing discounts with zero and using previous known values for product-level variables.

To avoid data leakage, columns related to same-day sales outcomes were removed from the feature set, including revenue, profit and other variables that may not be available before the prediction date.

The final output of this notebook is a feature-ready dataset saved as `modeling_dataset.csv`, which will be used in the next step to train and evaluate machine learning forecasting models.
